<a href="https://colab.research.google.com/github/aparna-2001/GATE_PPI/blob/main/merge_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd



In [ ]:
# Load all three
positives    = pd.read_parquet('/content/positives_string.parquet')
negatome     = pd.read_parquet('/content/combined_stringent.parquet')
random_negs  = pd.read_parquet('/content/negatives_random.parquet')

# Check current columns of each
print("Positives columns:   ", positives.columns.tolist())
print("Negatome columns:    ", negatome.columns.tolist())
print("Random neg columns:  ", random_negs.columns.tolist())

Positives columns:    ['protein1', 'protein2', 'neighborhood', 'fusion', 'cooccurence', 'coexpression', 'experimental', 'database', 'textmining', 'combined_score']
Negatome columns:     ['protein1', 'protein2']
Random neg columns:   ['protein1', 'protein2', 'label', 'source']


In [ ]:
# Positives — keep only protein1, protein2, add label and source
positives_clean = positives[['protein1', 'protein2']].copy()
positives_clean['label']  = 1
positives_clean['source'] = 'STRING_experimental'

# Negatome — already has protein1, protein2, add label and source
negatome_clean = negatome[['protein1', 'protein2']].copy()
negatome_clean['label']  = 0
negatome_clean['source'] = 'Negatome_combined_stringent'

# Random negatives — already correct, just confirm
random_clean = random_negs[['protein1', 'protein2', 'label', 'source']].copy()

# Verify all three now look identical in structure
print("Positives:")
print(positives_clean.head(2))
print(f"Shape: {positives_clean.shape}\n")

print("Negatome:")
print(negatome_clean.head(2))
print(f"Shape: {negatome_clean.shape}\n")

print("Random negatives:")
print(random_clean.head(2))
print(f"Shape: {random_clean.shape}")

Positives:
               protein1              protein2  label               source
0  9606.ENSP00000000233  9606.ENSP00000306010      1  STRING_experimental
1  9606.ENSP00000000233  9606.ENSP00000440005      1  STRING_experimental
Shape: (10405, 4)

Negatome:
  protein1 protein2  label                       source
0   Q6ZNK6   Q9Y4K3      0  Negatome_combined_stringent
1   Q9NR31   Q15797      0  Negatome_combined_stringent
Shape: (6136, 4)

Random negatives:
               protein1              protein2  label           source
0  9606.ENSP00000356669  9606.ENSP00000379350      0  random_negative
1  9606.ENSP00000333283  9606.ENSP00000434359      0  random_negative
Shape: (25079, 4)


In [ ]:
# Concatenate all three into one unified dataframe
pairs_combined = pd.concat(
    [positives_clean, negatome_clean, random_clean],
    ignore_index=True    # reindex from 0 — don't keep original indices
)

print(f"Total rows after merge: {len(pairs_combined):,}")
print(f"\nLabel distribution:")
print(pairs_combined['label'].value_counts())
print(f"\nSource distribution:")
print(pairs_combined['source'].value_counts())

Total rows after merge: 41,620

Label distribution:
label
0    31215
1    10405
Name: count, dtype: int64

Source distribution:
source
random_negative                25079
STRING_experimental            10405
Negatome_combined_stringent     6136
Name: count, dtype: int64


In [ ]:
# Check 1 — exact duplicate rows (same protein1, protein2, label, source)
exact_duplicates = pairs_combined.duplicated().sum()
print(f"Exact duplicate rows: {exact_duplicates}")

# Check 2 — duplicate pairs regardless of label or source
# (same protein1, protein2 appearing more than once)
pair_duplicates = pairs_combined.duplicated(
    subset=['protein1', 'protein2']
).sum()
print(f"Duplicate pairs (ignoring label/source): {pair_duplicates}")

# Check 3 — contradictory labels
# (same pair appearing as both label=1 and label=0)
pair_counts = pairs_combined.groupby(
    ['protein1', 'protein2']
)['label'].nunique()
contradictory = (pair_counts > 1).sum()
print(f"Contradictory labels (same pair, different label): {contradictory}")

Exact duplicate rows: 328
Duplicate pairs (ignoring label/source): 328
Contradictory labels (same pair, different label): 0


In [ ]:
before = len(pairs_combined)

# Keep first occurrence, drop subsequent duplicates
pairs_combined = pairs_combined.drop_duplicates(
    subset=['protein1', 'protein2'],
    keep='first'
).reset_index(drop=True)

after = len(pairs_combined)

print(f"Rows before: {before:,}")
print(f"Rows removed: {before - after:,}")
print(f"Rows after: {after:,}")
print(f"\nLabel distribution after dedup:")
print(pairs_combined['label'].value_counts())
print(f"\nSource distribution after dedup:")
print(pairs_combined['source'].value_counts())

Rows before: 41,620
Rows removed: 328
Rows after: 41,292

Label distribution after dedup:
label
0    30887
1    10405
Name: count, dtype: int64

Source distribution after dedup:
source
random_negative                25079
STRING_experimental            10405
Negatome_combined_stringent     5808
Name: count, dtype: int64


In [ ]:
# Final sanity check
print(f"Total pairs:       {len(pairs_combined):,}")
print(f"Positives:         {(pairs_combined['label']==1).sum():,}")
print(f"Negatives:         {(pairs_combined['label']==0).sum():,}")
print(f"Ratio:             {(pairs_combined['label']==0).sum() / (pairs_combined['label']==1).sum():.4f}")
print(f"Any nulls:         {pairs_combined.isnull().sum().sum()}")
print(f"Any duplicates:    {pairs_combined.duplicated(subset=['protein1','protein2']).sum()}")
print(f"\nFinal structure:")
print(pairs_combined.head(3))
print("...")
print(pairs_combined.tail(3))

Total pairs:       41,292
Positives:         10,405
Negatives:         30,887
Ratio:             2.9685
Any nulls:         0
Any duplicates:    0

Final structure:
               protein1              protein2  label               source
0  9606.ENSP00000000233  9606.ENSP00000306010      1  STRING_experimental
1  9606.ENSP00000000233  9606.ENSP00000440005      1  STRING_experimental
2  9606.ENSP00000000412  9606.ENSP00000438085      1  STRING_experimental
...
                   protein1              protein2  label           source
41289  9606.ENSP00000262187  9606.ENSP00000481152      0  random_negative
41290  9606.ENSP00000360365  9606.ENSP00000452549      0  random_negative
41291  9606.ENSP00000220751  9606.ENSP00000239906      0  random_negative


In [ ]:
import json
from datetime import datetime

# Save
pairs_combined.to_csv('/content/pairs_combined.tsv', sep='\t', index=False)
pairs_combined.to_parquet('/content/pairs_combined.parquet', index=False)

# Audit trail
metadata = {
    "date_created":         datetime.today().strftime('%Y-%m-%d'),
    "total_pairs":          len(pairs_combined),
    "positives":            int((pairs_combined['label']==1).sum()),
    "negatives":            int((pairs_combined['label']==0).sum()),
    "ratio":                round((pairs_combined['label']==0).sum() /
                                  (pairs_combined['label']==1).sum(), 4),
    "duplicates_removed":   328,
    "sources": {
        "STRING_experimental":           int((pairs_combined['source']=='STRING_experimental').sum()),
        "Negatome_combined_stringent":   int((pairs_combined['source']=='Negatome_combined_stringent').sum()),
        "random_negative":               int((pairs_combined['source']=='random_negative').sum())
    },
    "id_space_note": "STRING and random_negative use Ensembl IDs (9606.ENSPxxx). Negatome uses UniProt IDs. Reconciliation happens in Step 5.",
    "checks_passed": {
        "exact_duplicates_before_removal":  328,
        "contradictory_labels":             0,
        "nulls":                            0,
        "duplicates_after_removal":         0
    }
}

with open('/content/pairs_combined_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved successfully")
print(json.dumps(metadata, indent=2))

Saved successfully
{
  "date_created": "2026-06-17",
  "total_pairs": 41292,
  "positives": 10405,
  "negatives": 30887,
  "ratio": 2.9685,
  "duplicates_removed": 328,
  "sources": {
    "STRING_experimental": 10405,
    "Negatome_combined_stringent": 5808,
    "random_negative": 25079
  },
  "id_space_note": "STRING and random_negative use Ensembl IDs (9606.ENSPxxx). Negatome uses UniProt IDs. Reconciliation happens in Step 5.",
  "checks_passed": {
    "exact_duplicates_before_removal": 328,
    "contradictory_labels": 0,
    "nulls": 0,
    "duplicates_after_removal": 0
  }
}
